### Tín Hiệu Mua (Buy Signal): Một tín hiệu mua được xác định khi MA nhanh (MA_Fast) cao hơn hoặc bằng MA chậm (MA_Slow), và ATR hiện tại nhỏ hơn hoặc bằng ngưỡng ATR đã đặt. 
### Tín Hiệu Bán (Sell Signal): Một tín hiệu bán được xác định khi MA nhanh thấp hơn MA chậm, và ATR hiện tại cũng nhỏ hơn hoặc bằng ngưỡng ATR đã đặt.

# Forex, Timeframe m1


# B1: Lay duoc du lieu

In [48]:
# Import duoc Common 
import sys 
sys.path.append('../Common')
import CommonMT5
import MetaTrader5 as mt5
from datetime import datetime, timedelta

symbol = 'EURUSD.sml'
from_date = datetime.now().strftime('%Y-%m-%d')
to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')

timeframe = mt5.TIMEFRAME_M1
data = CommonMT5.CommonMT5.loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)
data

,Datetime,Open,High,Low,Close,Volume
0,2025-10-02 17:00:00,1.17226,1.17226,1.17182,1.17195,199
1,2025-10-02 17:01:00,1.17195,1.17206,1.17184,1.17199,156
2,2025-10-02 17:02:00,1.17199,1.17199,1.17148,1.17151,279
3,2025-10-02 17:03:00,1.17151,1.17175,1.17145,1.17163,200
4,2025-10-02 17:04:00,1.17160,1.17162,1.17135,1.17144,209
...,...,...,...,...,...,...
1418,2025-10-03 16:44:00,1.17424,1.17440,1.17424,1.17435,194
1419,2025-10-03 16:45:00,1.17434,1.17444,1.17422,1.17425,300
1420,2025-10-03 16:46:00,1.17425,1.17428,1.17401,1.17419,241
1421,2025-10-03 16:47:00,1.17416,1.17432,1.17395,1.17432,282


# B2: Xu ly du lieu

In [49]:
# MA_Fast 10, MA_Slow 200, ATR 
import talib
data['MA_Fast'] = talib.SMA(data['Close'], timeperiod=10)
data['MA_Slow'] = talib.SMA(data['Close'], timeperiod=200)
data['ATR'] = talib.ATR(data['High'], data['Low'], data['Close'], timeperiod=14)
data

,Datetime,Open,High,Low,Close,Volume,MA_Fast,MA_Slow,ATR
0,2025-10-02 17:00:00,1.17226,1.17226,1.17182,1.17195,199,NaN,NaN,NaN
1,2025-10-02 17:01:00,1.17195,1.17206,1.17184,1.17199,156,NaN,NaN,NaN
2,2025-10-02 17:02:00,1.17199,1.17199,1.17148,1.17151,279,NaN,NaN,NaN
3,2025-10-02 17:03:00,1.17151,1.17175,1.17145,1.17163,200,NaN,NaN,NaN
4,2025-10-02 17:04:00,1.17160,1.17162,1.17135,1.17144,209,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1418,2025-10-03 16:44:00,1.17424,1.17440,1.17424,1.17435,194,1.173991,1.173661,0.000196
1419,2025-10-03 16:45:00,1.17434,1.17444,1.17422,1.17425,300,1.174028,1.173664,0.000198
1420,2025-10-03 16:46:00,1.17425,1.17428,1.17401,1.17419,241,1.174068,1.173667,0.000203
1421,2025-10-03 16:47:00,1.17416,1.17432,1.17395,1.17432,282,1.174116,1.173671,0.000215


B2.1 / 2.2: Buy_Signal, Sell_Signal

In [50]:
atr_threshold = 0.00022
data['Buy_Signal'] = (data['MA_Fast'] >= data['MA_Slow']) & (data['ATR'] <= atr_threshold)
data['Sell_Signal'] = (data['MA_Fast'] < data['MA_Slow']) & (data['ATR'] <= atr_threshold)
data

,Datetime,Open,High,Low,Close,Volume,MA_Fast,MA_Slow,ATR,Buy_Signal,Sell_Signal
0,2025-10-02 17:00:00,1.17226,1.17226,1.17182,1.17195,199,NaN,NaN,NaN,False,False
1,2025-10-02 17:01:00,1.17195,1.17206,1.17184,1.17199,156,NaN,NaN,NaN,False,False
2,2025-10-02 17:02:00,1.17199,1.17199,1.17148,1.17151,279,NaN,NaN,NaN,False,False
3,2025-10-02 17:03:00,1.17151,1.17175,1.17145,1.17163,200,NaN,NaN,NaN,False,False
4,2025-10-02 17:04:00,1.17160,1.17162,1.17135,1.17144,209,NaN,NaN,NaN,False,False
...,...,...,...,...,...,...,...,...,...,...,...
1418,2025-10-03 16:44:00,1.17424,1.17440,1.17424,1.17435,194,1.173991,1.173661,0.000196,True,False
1419,2025-10-03 16:45:00,1.17434,1.17444,1.17422,1.17425,300,1.174028,1.173664,0.000198,True,False
1420,2025-10-03 16:46:00,1.17425,1.17428,1.17401,1.17419,241,1.174068,1.173667,0.000203,True,False
1421,2025-10-03 16:47:00,1.17416,1.17432,1.17395,1.17432,282,1.174116,1.173671,0.000215,True,False


B2.3 Da sang Redis

In [51]:
# Day sang Redis 0->5: CK, 6-10: FX, 11-15: Crypto

import redis
from datetime import datetime
import numpy as np
import pandas as pd

r = redis.Redis(host='localhost', port=6379, db=6) 
# Tạo hash key
hash_key = 'Buoi 8_OG MA, ATR'
last_record = data.iloc[-2].copy()  # Sử dụng .copy() để tránh warning

# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
    # Thêm Insertdate vào record trước khi lưu
    last_record['Insertdate'] = datetime.now().isoformat()
    
    for field, value in last_record.to_dict().items():
        # Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
        if isinstance(value, pd.Timestamp):
            value = value.isoformat()
        elif isinstance(value, (int, np.uint64)):
            value = str(value)
        r.hset(hash_key, field, value)  
        r.hset(hash_key, 'Symbol', symbol)
    
    print("Tín hiệu giao dịch:")
    print(last_record)   
else:
    print("Không có tín hiệu giao dịch:")
    print(last_record)   
    print('Không có tín hiệu!')

Tín hiệu giao dịch:
Datetime              2025-10-03 16:47:00
Open                              1.17416
High                              1.17432
Low                               1.17395
Close                             1.17432
Volume                                282
MA_Fast                          1.174116
MA_Slow                          1.173671
ATR                              0.000215
Buy_Signal                           True
Sell_Signal                         False
Insertdate     2025-10-03T20:48:30.209933
Name: 1421, dtype: object


B3. Dat vao Auto Trade